In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

c:\Users\GAYATHRI DEVI\anaconda3\envs\rag_ml_chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# from dotenv import load_dotenv

# load_dotenv()

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",  
#     temperature=0.7
# )

# response = llm.invoke("Explain LangChain simply")
# print(response.content)

Imagine you have a super-smart, but somewhat isolated, expert (that's your Large Language Model or LLM, like the brain behind ChatGPT).

This expert is brilliant at understanding language, writing, and answering questions based on what they've learned from their training.

**Here's the problem with our super-smart expert:**

1.  **No Real-time Info:** They don't know what happened five minutes ago, or what's in your private company documents.
2.  **No Tools:** They can't browse the internet, use a calculator, look up a specific file, or send an email.
3.  **One-Shot Thinking:** They can only process one request at a time and don't remember previous conversations very well.
4.  **Big Tasks are Hard:** If you give them a really complex, multi-step task, they might get overwhelmed.

---

**LangChain is like a super-assistant or a project manager for your super-smart expert (the LLM).**

It gives the LLM everything it needs to become much more powerful and useful in real-world applications

In [3]:
def file_loader(path):
  loader = DirectoryLoader(
    path, glob="*.pdf", loader_cls=PyPDFLoader
  )
  documents = loader.load()
  return documents

In [4]:
extracted_docs = file_loader(r"Data/")

In [5]:
def chunking_data(data):
  split_data = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 50)
  chunk_data = split_data.split_documents(data)
  return chunk_data

In [6]:
chunk_data = chunking_data(extracted_docs)
len(chunk_data)

1680

In [8]:
chunk_data[60].page_content

'Safari® Books Online\nSafari Books Online is an on-demand digital library that deliv‐\ners expert content in both book and video form from the\nworld’s leading authors in technology and business.\nTechnology professionals, software developers, web designers, and business and crea‐\ntive professionals use Safari Books Online as their primary resource for research,\nproblem solving, learning, and certification training.'

In [9]:
def get_embedding():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [10]:
embedding = get_embedding()

C:\Users\GAYATHRI DEVI\AppData\Local\Temp\ipykernel_22496\2649802877.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
c:\Users\GAYATHRI DEVI\anaconda3\envs\rag_ml_chatbot\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GAYATHRI DEVI\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be dis

In [11]:
docs = FAISS.from_documents(documents=chunk_data, embedding=embedding )
docs

In [12]:
retriver = docs.as_retriever(search_type="similarity", search_kwargs={"k": 3})
output = retriver.invoke("What is Supervised Machine Learning")
output

[Document(id='3ca11964-78cd-4432-94f2-248b22689385', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2016-09-21T13:04:39+00:00', 'author': 'Andreas C. Müller and Sarah Guido', 'title': 'Introduction to Machine Learning with Python', 'trapped': '/False', 'moddate': '2020-08-19T07:09:16+02:00', 'source': 'Data\\Machine Learning.pdf', 'total_pages': 392, 'page': 15, 'page_label': '2'}, page_content='spam.\nMachine learning algorithms that learn from input/output pairs are called supervised\nlearning algorithms because a “teacher” provides supervision to the algorithms in the\nform of the desired outputs for each example that they learn from. While creating a\ndataset of inputs and outputs is often a laborious manual process, supervised learning\nalgorithms are well understood and their performance is easy to measure. If your'),
 Documen

In [13]:
from dotenv import load_dotenv
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  
    temperature=0.7
)

In [18]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

In [19]:
system_prompt = (
"You are an expert Data Scientist assistat of qestion-answering tasks."
"Use the following pieces of retrieved context to answer "
"the question. If you don't find any related context then say that you "
"don't know. Do not give any halusinating answer of this. Use the three sentece maximum and keep the "
"answer concise."
"\n\n"
"{context}"
 )

chat_prompt = ChatPromptTemplate.from_messages([
  ("system", system_prompt),
  ("user", "{input}" )]
)

In [ ]:
stuff_chain =create_stuff_documents_chain(llm, chat_prompt)
retriver_chain = create_retrieval_chain(retriver, stuff_chain)

question = "What is Machine Learnig?"
response_dict = retriver_chain.invoke({"input" : question})

response_dict['answer']

'Machine learning is about extracting knowledge from data. It is a research field at the intersection of statistics, artificial intelligence, and computer science, also known as predictive analytics or statistical learning.'

In [21]:
question = "What is Tensorflow?"
response_dict = retriver_chain.invoke({"input" : question})
response_dict['answer']

'TensorFlow is a well-established deep learning library and framework for Python users. It provides a flexible interface to build neural networks and track progress in deep learning research. Additionally, it supports the use of high-performance graphics processing units (GPUs) to accelerate computations.'